In [0]:
dbutils.widgets.removeAll()

In [0]:
import logging
from datetime import datetime
import os

# Create logs directory if it doesn't exist
log_dir = "/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing/logs"
dbutils.fs.mkdirs(f"file:{log_dir}")

# Create log filename with timestamp
log_filename = f"{log_dir}/provider_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename.replace('file:', '')),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("=" * 60)
logger.info("Provider Silver Layer Processing Started")
logger.info(f"Log file: {log_filename}")
logger.info("=" * 60)

In [0]:
try:
    dbutils.widgets.text("ClientContainer", "claimspan", "Client Container / Catalog Name")
    client_container = dbutils.widgets.get("ClientContainer").strip()
except Exception:
    client_container = "claimspan"

# Wrap catalog name in backticks for safety (e.g. `274`)
safe_catalog = f"`{client_container}`"

sourcePath = f"/Volumes/{client_container}/bronze/provider_consolidated"
silverProviderBridgeTable = f"{safe_catalog}.silver.silver_providerpersonbridge"
silverProviderTable = f"{safe_catalog}.silver.silver_provider"
providerSpecialtyDatasetPath = f"{safe_catalog}.silver.ref_careprecise_taxonomy"
carePreciseTaxoPath = f"{safe_catalog}.silver.ref_careprecise_taxonomy"
credentialingPath = f"{safe_catalog}.silver.ref_credentialing"
haiReportingListPath = f"{safe_catalog}.silver.ref_hai_reporting"
hospitalAffiliationPath = f"{safe_catalog}.silver.ref_hospital_affiliation" 

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Source Path: {sourcePath}")
print(f"Silver Person Bridge Table: {silverProviderBridgeTable}")
print(f"Silver Provider Table: {silverProviderTable}")
print("="*60)

logger.info("Configuration loaded:")
logger.info(f"  Source Path: {sourcePath}")
logger.info(f"  Silver Person Bridge Table: {silverProviderBridgeTable}")
logger.info(f"  Silver Provider Table: {silverProviderTable}")

In [0]:
def table_exists(tableToCheck):
    """Check if a table or path exists and can be read"""
    try:
        if '/' in tableToCheck:
            # It is a path
            spark.read.format("parquet").load(tableToCheck)
        else:
            # It is a registered table name
            spark.table(tableToCheck)
        logger.info(f"Data check: {tableToCheck} - Data exists")
        return True
    except Exception as e:
        logger.warning(f"Data check: {tableToCheck} - Data does not exist or is inaccessible: {str(e)}")
        return False

In [0]:
from pyspark.sql.functions import date_format, sha2, concat_ws, col, row_number, expr
from pyspark.sql.window import Window

In [0]:
srcsql = f"""
WITH consolidateProvider1 as (
  SELECT 
    brdg.ESAIInternalProviderID, brdg.uniqueRecord, prov.ClientID, prov.FileID, prov.LoadDateTime, 
    prov.FileLayoutID, prov.FileLayoutDescription, prov.identifier_providerID, prov.name_family, prov.name_given_middle_initial, 
    prov.name_given_first, 
    coalesce(prov.identifier_taxonomyCode1, cpt1.Taxo) AS TaxonomyCode1, 
    coalesce(s1.Specialty, prov.identifier_hpSpecialtyCode1) AS HpSpecialtyCode1, 
    coalesce(right(concat('00', cast(s1.CMSSpecialtyCode as string)), 2), prov.identifier_advProviderSpecialtyCode1) AS ADVProviderSpecialtyCode1, 
    coalesce(prov.identifier_taxonomyCode2, cpt2.Taxo) AS TaxonomyCode2, 
    coalesce(s2.Specialty, prov.identifier_hpSpecialtyCode2) AS HpSpecialtyCode2, 
    coalesce(right(concat('00', cast(s2.CMSSpecialtyCode as string)), 2), prov.identifier_advProviderSpecialtyCode2) AS ADVProviderSpecialtyCode2, 
    coalesce(prov.identifier_taxonomyCode3, cpt3.Taxo) AS TaxonomyCode3, 
    coalesce(s3.Specialty, prov.identifier_hpSpecialtyCode3) AS HpSpecialtyCode3, 
    coalesce(right(concat('00', cast(s3.CMSSpecialtyCode as string)), 2), prov.identifier_advProviderSpecialtyCode3) AS ADVProviderSpecialtyCode3, 
    coalesce(prov.identifier_taxonomyCode4, cpt4.Taxo) AS TaxonomyCode4, 
    coalesce(s4.Specialty, prov.identifier_hpSpecialtyCode4) AS HpSpecialtyCode4, 
    coalesce(right(concat('00', cast(s4.CMSSpecialtyCode as string)), 2), prov.identifier_advProviderSpecialtyCode4) AS ADVProviderSpecialtyCode4, 
    coalesce(prov.identifier_taxonomyCode5, cpt5.Taxo) AS TaxonomyCode5, 
    coalesce(s5.Specialty, prov.identifier_hpSpecialtyCode5) AS HpSpecialtyCode5, 
    coalesce(right(concat('00', cast(s5.CMSSpecialtyCode as string)), 2), prov.identifier_advProviderSpecialtyCode5) AS ADVProviderSpecialtyCode5, 
    brdg.identifier_npi, prov.extension_prescribePrivilege, brdg.identifier_providerDEA, prov.identifier_payerID, 
    coalesce(cred.Contracted, prov.extension_contracted, 'N') AS Contracted, 
    coalesce(hai.ProviderHAI, prov.identifier_providerHAI, 'N') AS ProviderHAI, 
    coalesce(hosp.HospitalID, prov.identifier_hospitalID) AS HospitalID, 
    coalesce(prov.extension_excludeFromProviderReporting, 'N') AS ExcludeFromProviderReporting, 
    prov.identifier_altProvReporting1, prov.identifier_altProvReporting2, prov.identifier_altProvReporting3, 
    prov.identifier_altProvReporting4, prov.identifier_altProvReporting5, prov.identifier_altProvReporting6, 
    prov.identifier_altProvReporting7, prov.identifier_altProvReporting8, prov.identifier_altProvReporting9, 
    prov.identifier_altProvReporting10, brdg.pmup, brdg.isCurrentPMUP
  FROM consolidateProvider prov
  INNER JOIN silverProviderBridge brdg 
    ON prov.UniqueRecord = brdg.uniqueRecord 
    AND prov.FileLayoutID = brdg.fileLayoutID 
    AND brdg.isCurrentPMUP = 1
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt1 ON prov.identifier_taxonomyCode1 = cpt1.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s1 ON prov.identifier_taxonomyCode1 = s1.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt2 ON prov.identifier_taxonomyCode2 = cpt2.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s2 ON prov.identifier_taxonomyCode2 = s2.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt3 ON prov.identifier_taxonomyCode3 = cpt3.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s3 ON prov.identifier_taxonomyCode3 = s3.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt4 ON prov.identifier_taxonomyCode4 = cpt4.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s4 ON prov.identifier_taxonomyCode4 = s4.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt5 ON prov.identifier_taxonomyCode5 = cpt5.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s5 ON prov.identifier_taxonomyCode5 = s5.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_credentialing cred ON prov.identifier_providerID = cred.ProviderID
  LEFT JOIN {safe_catalog}.silver.ref_hai_reporting hai ON prov.identifier_providerID = hai.ProviderID
  LEFT JOIN {safe_catalog}.silver.ref_hospital_affiliation hosp ON prov.identifier_providerID = hosp.ProviderID
)
SELECT 
  ESAIInternalProviderID, uniqueRecord, ClientID, FileID, LoadDateTime, FileLayoutID, 
  FileLayoutDescription, identifier_providerID, name_family, name_given_middle_initial, name_given_first, 
  TaxonomyCode1 AS taxonomyCode1, HpSpecialtyCode1 AS hpSpecialtyCode1, ADVProviderSpecialtyCode1 AS advProviderSpecialtyCode1, 
  TaxonomyCode2 AS taxonomyCode2, HpSpecialtyCode2 AS hpSpecialtyCode2, ADVProviderSpecialtyCode2 AS advProviderSpecialtyCode2, 
  TaxonomyCode3 AS taxonomyCode3, HpSpecialtyCode3 AS hpSpecialtyCode3, ADVProviderSpecialtyCode3 AS advProviderSpecialtyCode3, 
  TaxonomyCode4 AS taxonomyCode4, HpSpecialtyCode4 AS hpSpecialtyCode4, ADVProviderSpecialtyCode4 AS advProviderSpecialtyCode4, 
  TaxonomyCode5 AS taxonomyCode5, HpSpecialtyCode5 AS hpSpecialtyCode5, ADVProviderSpecialtyCode5 AS advProviderSpecialtyCode5, 
  identifier_npi, extension_prescribePrivilege AS extension_isPrescribePrivilege, identifier_providerDEA, identifier_payerID, Contracted AS extension_isContracted, 
  ProviderHAI AS extension_providerHAI, HospitalID AS identifier_hospitalID, ExcludeFromProviderReporting AS extension_isExcludedFromProviderReporting, 
  identifier_altProvReporting1 AS identifier_alternateKey1, identifier_altProvReporting2 AS identifier_alternateKey2, identifier_altProvReporting3 AS identifier_alternateKey3, 
  identifier_altProvReporting4 AS identifier_alternateKey4, identifier_altProvReporting5 AS identifier_alternateKey5, identifier_altProvReporting6 AS identifier_alternateKey6, 
  identifier_altProvReporting7 AS identifier_alternateKey7, identifier_altProvReporting8 AS identifier_alternateKey8, identifier_altProvReporting9 AS identifier_alternateKey9, 
  identifier_altProvReporting10 AS identifier_alternateKey10, pmup, isCurrentPMUP,
  sha2(concat(
    IfNull(ESAIInternalProviderID,""), "|", IfNull(uniqueRecord,""), "|", 
    IfNull(ClientID,""), "|", IfNull(CAST(FileID AS STRING),""), "|", 
    IfNull(CAST(LoadDateTime AS STRING),""), "|", IfNull(CAST(FileLayoutID AS STRING),""), "|", 
    IfNull(FileLayoutDescription,""), "|", IfNull(identifier_providerID,""), "|", 
    IfNull(name_family,""), "|", IfNull(name_given_middle_initial,""), "|", IfNull(name_given_first,""), "|", 
    IfNull(TaxonomyCode1,""), "|", IfNull(HpSpecialtyCode1,""), "|", IfNull(ADVProviderSpecialtyCode1,""), "|", 
    IfNull(TaxonomyCode2,""), "|", IfNull(HpSpecialtyCode2,""), "|", IfNull(ADVProviderSpecialtyCode2,""), "|", 
    IfNull(TaxonomyCode3,""), "|", IfNull(HpSpecialtyCode3,""), "|", IfNull(ADVProviderSpecialtyCode3,""), "|", 
    IfNull(TaxonomyCode4,""), "|", IfNull(HpSpecialtyCode4,""), "|", IfNull(ADVProviderSpecialtyCode4,""), "|", 
    IfNull(TaxonomyCode5,""), "|", IfNull(HpSpecialtyCode5,""), "|", IfNull(ADVProviderSpecialtyCode5,""), "|", 
    IfNull(identifier_npi,""), "|", IfNull(extension_prescribePrivilege,""), "|", 
    IfNull(identifier_providerDEA,""), "|", IfNull(identifier_payerID,""), "|", IfNull(Contracted,""), "|", 
    IfNull(ProviderHAI,""), "|", IfNull(HospitalID,""), "|", IfNull(ExcludeFromProviderReporting,""), "|", 
    IfNull(identifier_altProvReporting1,""), "|", IfNull(identifier_altProvReporting2,""), "|", 
    IfNull(identifier_altProvReporting3,""), "|", IfNull(identifier_altProvReporting4,""), "|", 
    IfNull(identifier_altProvReporting5,""), "|", IfNull(identifier_altProvReporting6,""), "|", 
    IfNull(identifier_altProvReporting7,""), "|", IfNull(identifier_altProvReporting8,""), "|", 
    IfNull(identifier_altProvReporting9,""), "|", IfNull(identifier_altProvReporting10,""), "|", 
    IfNull(pmup,""), "|", IfNull(CAST(isCurrentPMUP AS STRING),"")
  ), 256) AS hashKey 
FROM consolidateProvider1
"""

In [0]:
logger.info("MAIN EXECUTION STARTED")
print(f"Source Path: {sourcePath}")
print(f"Person Bridge: {silverProviderBridgeTable}")

if table_exists(sourcePath) and table_exists(silverProviderBridgeTable):
    print("Loading Bronze Provider data...")
    dfconsolidateProvSrc = spark.read.format("delta").load(sourcePath)
    
    print("Loading Person Bridge table...")
    dfsilverProvBrdg = spark.table(silverProviderBridgeTable)
    
    # Run row fingerprints
    windowPartition = Window.partitionBy(col("FileID")).orderBy(col("RecordHash").desc())
    dfconsolidateProv = dfconsolidateProvSrc.distinct() \
        .withColumn("RecordHash", sha2(concat_ws("||", *dfconsolidateProvSrc.columns), 256)) \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-", col("FileID"), col("RowNumber"))) \
        .withColumn("FileLayoutID", expr("try_cast(FileLayoutID as int)"))
        
    dfconsolidateProv.createOrReplaceTempView("consolidateProvider")
    dfsilverProvBrdg.createOrReplaceTempView("silverProviderBridge")
    
    print("Joining Provider profiles with Person Bridge...")
    dfsrc = spark.sql(srcsql)
    
    recordCount = dfsrc.count()
    print(f"Found {recordCount} records to write")
    
    if recordCount > 0:
        print(f"Writing to Silver Provider Table: {silverProviderTable}")
        spark.sql(f"DROP TABLE IF EXISTS {silverProviderTable}")
        dfsrc.write.format("delta").mode("overwrite").saveAsTable(silverProviderTable)
        print("Silver Provider table created successfully!")
    else:
        print("No records found to write.")
else:
    print("Bronze source or Person Bridge table does not exist. Skipping execution.")
    logger.error("Source or Bridge table missing")

In [0]:
%sql
DESCRIBE TABLE claimspan.silver.silver_providerpersonbridge;

In [0]:
display(spark.read.format('delta').load("/Volumes/claimspan/bronze/provider_consolidated"))